# Lumis 12: Train model v2 (clinical + phone photos)

Model v1 learned only DermNet's clinical photos. v2 **starts from v1** and keeps training on DermNet's training photos
**plus** SCIN's training photos (self-taken phone photos, notebook 11). This is fine-tuning for a new photo style
(domain adaptation): v1 already knows the conditions; v2 learns what they look like on a phone.

Settings fixed before training (no searching over them):
- SCIN training photos are shown **3 times** per epoch (`SCIN_REPEAT`), so the smaller phone set is not drowned out by DermNet.
- Same unfrozen layers as v1 (top 35% of EfficientNet-B0), batch-norm layers frozen, same augmentation, balanced class weights.
- Learning rate 3e-5 (lower than v1's 5e-5 because training starts from a trained model), up to 20 epochs,
  early stopping on validation loss over DermNet validation + SCIN validation (SCIN counted 3 times, as in training).

**Class balance (`CLASS_BALANCE`).** The first v2 run (`efficientnetb0v2_20261111_1501`) balanced classes over
DermNet and SCIN together (`'overall'`). Because 64% of SCIN photos are eczema, it learned a shortcut: "looks like a
phone photo, so eczema". SCIN validation accuracy rose, but tinea sensitivity on phone photos halved, and tinea
mistaken for eczema is the harmful direction (steroid creams make tinea worse). Run v2b (`'per_source'`, the default
now) balances classes **within each dataset**, so a phone photo of tinea counts as much as a phone photo of eczema.
SCIN weights are capped at 5 (scabies has only 8 SCIN training photos). Early stopping uses the same weights on
validation. Everything else is unchanged.

Test photos of both datasets are never loaded here.

**Runtime: GPU (T4).** About 30 to 60 minutes.

In [ ]:
SCIN_REPEAT = 3
CLASS_BALANCE = 'per_source'   # 'per_source' (v2b) or 'overall' (what the first v2 run used)
MAX_WEIGHT = 5.0               # cap on any photo's weight under 'per_source'
FINE_LR = 3e-5
EPOCHS = 20
UNFREEZE_TOP = 0.35
IMG_SIZE = 224
BATCH = 32
SEED = 42
V1_RUN = None          # None = the v1 final model (latest EfficientNet-B0 run on split v2)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/Lumis'

In [ ]:
import os, zipfile, json, time
import numpy as np, pandas as pd
import tensorflow as tf
import keras
from keras import layers

LOCAL = '/content/dermnet'
if not os.path.exists(LOCAL):
    with zipfile.ZipFile(f'{BASE}/data/dermnet.zip') as z:
        z.extractall(LOCAL)
if not os.path.exists('/content/scin'):
    with zipfile.ZipFile(f'{BASE}/data/scin_images.zip') as z:
        z.extractall('/content')

keras.utils.set_random_seed(SEED)
print('TensorFlow', tf.__version__, '| Keras', keras.__version__)
print('GPU:', tf.config.list_physical_devices('GPU') or 'NONE - switch the runtime to T4 GPU')

runs = pd.read_csv(f'{BASE}/exports/runs_log.csv')
if V1_RUN is None:
    V1_RUN = runs[(runs.backbone == 'efficientnetb0') & (runs.split == 'v2')].iloc[-1]['run']
v1 = runs[runs.run == V1_RUN].iloc[-1]
print('Starting from v1:', V1_RUN)

derm = pd.read_csv(f'{BASE}/data/lumis_split_v2.csv')
scin = pd.read_csv(f'{BASE}/data/scin_split_v1.csv', dtype={'case_id': str})
CLASSES = sorted(derm.lumis_class.unique())
IDX = {c: i for i, c in enumerate(CLASSES)}
d_train, d_val = derm[derm.split == 'train'], derm[derm.split == 'val'].reset_index(drop=True)
s_train, s_val = scin[scin.split == 'train'], scin[scin.split == 'val'].reset_index(drop=True)
# The test splits of both datasets are deliberately never used in this notebook
print(f'DermNet train {len(d_train)}, val {len(d_val)} | SCIN train {len(s_train)} photos '
      f'({s_train.case_id.nunique()} cases), val {len(s_val)} photos ({s_val.case_id.nunique()} cases)')

## 1. Data

In [ ]:
def load(path, label):
    img = tf.io.decode_image(tf.io.read_file(path), channels=3, expand_animations=False)
    img = tf.image.resize(img, (IMG_SIZE, IMG_SIZE))
    return tf.cast(img, tf.uint8), label

augment = keras.Sequential([
    layers.RandomFlip('horizontal_and_vertical'),
    layers.RandomRotation(0.1),
    layers.RandomZoom(0.15),
    layers.RandomBrightness(0.15, value_range=(0, 255)),
    layers.RandomContrast(0.15),
])
preprocess = keras.applications.efficientnet.preprocess_input

def paths_labels(df, root):
    return [os.path.join(root, p) for p in df.path], [IDX[c] for c in df.lumis_class]

def make_ds(paths, labels, training, weights=None):
    weights = np.ones(len(paths), np.float32) if weights is None else np.asarray(weights, np.float32)
    ds = tf.data.Dataset.from_tensor_slices((paths, labels, weights))
    ds = ds.map(lambda p, y, w: (*load(p, y), w), num_parallel_calls=tf.data.AUTOTUNE).cache()
    if training:
        ds = ds.shuffle(len(paths), seed=SEED)
    ds = ds.map(lambda x, y, w: (tf.cast(x, tf.float32), y, w))
    if training:
        ds = ds.map(lambda x, y, w: (augment(x, training=True), y, w), num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.map(lambda x, y, w: (preprocess(x), y, w), num_parallel_calls=tf.data.AUTOTUNE)
    return ds.batch(BATCH).prefetch(tf.data.AUTOTUNE)

K = len(CLASSES)
def balanced(labels, cap=None):
    labels = np.asarray(labels)
    w = len(labels) / (K * np.maximum(np.bincount(labels, minlength=K), 1))
    if cap is not None:
        w = np.minimum(w, cap)
    return w[labels]

def photo_weights(d_labels, s_labels):
    """Weights for DermNet photos followed by SCIN photos (SCIN already repeated)."""
    if CLASS_BALANCE == 'per_source':
        return np.concatenate([balanced(d_labels), balanced(s_labels, MAX_WEIGHT)])
    return balanced(list(d_labels) + list(s_labels))          # 'overall' = sklearn 'balanced' class weights

dp, dl = paths_labels(d_train, LOCAL)
sp, sl = paths_labels(s_train, '/content')
train_paths, train_labels = dp + sp * SCIN_REPEAT, dl + sl * SCIN_REPEAT
vdp, vdl = paths_labels(d_val, LOCAL)
vsp, vsl = paths_labels(s_val, '/content')
train_w = photo_weights(dl, sl * SCIN_REPEAT)
val_w = photo_weights(vdl, vsl * SCIN_REPEAT)
train_ds = make_ds(train_paths, train_labels, training=True, weights=train_w)
val_ds = make_ds(vdp + vsp * SCIN_REPEAT, vdl + vsl * SCIN_REPEAT, training=False, weights=val_w)

n_d = len(dp)
wt = pd.DataFrame({'class': [CLASSES[i] for i in train_labels], 'source': ['DermNet'] * n_d + ['SCIN'] * (len(train_labels) - n_d),
                   'weight': train_w})
print('Training photos per epoch:', len(train_paths), '| balance:', CLASS_BALANCE)
display(wt.groupby(['source', 'class']).weight.agg(['count', 'first', 'sum']).round(2)
          .rename(columns={'count': 'photos', 'first': 'weight each', 'sum': 'total weight'}))

## 2. Fine-tune from v1

In [ ]:
model = keras.models.load_model(v1.model_file)
base = next(l for l in model.layers if isinstance(l, keras.Model))
base.trainable = True
cut = int(len(base.layers) * (1 - UNFREEZE_TOP))
for layer in base.layers[:cut]:
    layer.trainable = False
for layer in base.layers:
    if isinstance(layer, layers.BatchNormalization):
        layer.trainable = False
print(f'Fine-tuning {len(base.layers) - cut} of {len(base.layers)} backbone layers ({base.name})')

RUN = f"efficientnetb0v2{'b' if CLASS_BALANCE == 'per_source' else ''}_{time.strftime('%Y%m%d_%H%M')}"
RUN_DIR = f'{BASE}/exports/runs/{RUN}'
os.makedirs(RUN_DIR, exist_ok=True)
CKPT = f'{BASE}/checkpoints/{RUN}.keras'

model.compile(optimizer=keras.optimizers.Adam(FINE_LR),
              loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True), metrics=['accuracy'])
hist = model.fit(train_ds, validation_data=val_ds, epochs=EPOCHS,
                 callbacks=[keras.callbacks.EarlyStopping(patience=4, restore_best_weights=True),
                            keras.callbacks.ReduceLROnPlateau(factor=0.3, patience=2, min_lr=1e-7),
                            keras.callbacks.ModelCheckpoint(CKPT, save_best_only=True)])
model.save(CKPT)
print('Run:', RUN, '| saved', CKPT)

In [ ]:
import matplotlib.pyplot as plt
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
for i, m in enumerate(['loss', 'accuracy']):
    ax[i].plot(hist.history[m], label='train')
    ax[i].plot(hist.history[f'val_{m}'], label='validation (DermNet + SCIN)')
    ax[i].set_title(m); ax[i].set_xlabel('epoch'); ax[i].legend()
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/training_curves.png', dpi=100)
plt.show()

## 3. v1 versus v2 on validation

DermNet validation: all photos. SCIN validation: one photo per case (the close-up), like a Lumis user.
Raw model output, before calibration (notebook 13).

In [ ]:
from sklearn.metrics import confusion_matrix, f1_score

def predict(m, paths):
    ds = make_ds(paths, [0] * len(paths), training=False)
    return m.predict(ds, verbose=0)

def scores(logits, y):
    pred = logits.argmax(1)
    cm = confusion_matrix(y, pred, labels=range(len(CLASSES)))
    sens = cm.diagonal() / np.maximum(cm.sum(1), 1)
    return {'accuracy': round(float((pred == y).mean()), 4),
            'macro_sensitivity': round(float(sens[cm.sum(1) > 0].mean()), 4),
            'macro_f1': round(float(f1_score(y, pred, average='macro', labels=range(len(CLASSES)), zero_division=0)), 4),
            **{f'sens_{c}': round(float(s), 3) for c, s in zip(CLASSES, sens)}}

s_eval = s_val[s_val.eval_photo].reset_index(drop=True)
y_d = np.array(vdl)
y_s = np.array([IDX[c] for c in s_eval.lumis_class])
eval_paths, _ = paths_labels(s_eval, '/content')

v2_d, v2_s = predict(model, vdp), predict(model, eval_paths)
v1_model = keras.models.load_model(v1.model_file)
v1_d, v1_s = np.load(f'{BASE}/exports/runs/{V1_RUN}/val_logits.npy'), predict(v1_model, eval_paths)

table = pd.DataFrame({'v1 DermNet val': scores(v1_d, y_d), 'v2 DermNet val': scores(v2_d, y_d),
                      'v1 SCIN val': scores(v1_s, y_s), 'v2 SCIN val': scores(v2_s, y_s)})
display(table)
print(f'SCIN validation: {len(s_eval)} cases', s_eval.lumis_class.value_counts().to_dict())

np.save(f'{RUN_DIR}/val_logits.npy', v2_d)
np.save(f'{RUN_DIR}/val_labels.npy', y_d)
np.save(f'{RUN_DIR}/scin_val_logits.npy', v2_s)
np.save(f'{RUN_DIR}/scin_val_labels.npy', y_s)
np.save(f'{RUN_DIR}/scin_val_logits_v1.npy', v1_s)
s_eval[['path', 'case_id', 'lumis_class', 'tone']].to_csv(f'{RUN_DIR}/scin_val_cases.csv', index=False)
table.to_csv(f'{RUN_DIR}/v1_vs_v2_validation.csv')

In [ ]:
d2, s2 = scores(v2_d, y_d), scores(v2_s, y_s)
summary = {
    'run': RUN, 'backbone': 'efficientnetb0', 'split': 'v2+scin', 'base_run': V1_RUN, 'img_size': IMG_SIZE,
    'batch': BATCH, 'seed': SEED, 'scin_repeat': SCIN_REPEAT, 'class_balance': CLASS_BALANCE, 'fine_lr': FINE_LR, 'unfreeze_top': UNFREEZE_TOP,
    'fine_epochs_run': len(hist.history['loss']), 'params': int(model.count_params()),
    'val_accuracy': d2['accuracy'], 'val_macro_f1': d2['macro_f1'], 'val_macro_sensitivity': d2['macro_sensitivity'],
    'scin_val_accuracy': s2['accuracy'], 'scin_val_macro_f1': s2['macro_f1'],
    'scin_val_macro_sensitivity': s2['macro_sensitivity'], 'model_file': CKPT,
}
json.dump(summary, open(f'{RUN_DIR}/summary.json', 'w'), indent=1)
LOG = f'{BASE}/exports/runs_log.csv'
pd.concat([pd.read_csv(LOG), pd.DataFrame([summary])]).to_csv(LOG, index=False)
print(json.dumps(summary, indent=1))